# Get ensemble transcriptions of the Daily Rainfall images

The daily rainfall ensemble transcriptions are a *separate* data source, stored
in their own SQLite database (`ensemble_transcriptions.sqlite`). Each JSON file
holds keys `Day 1` .. `Day 31` plus a `Totals` block; every key maps to 12
month slots (January-December) and each month slot has 5 ensemble member values.

This section performs a full rebuild of that database from all JSON files in the
`ensemble_transcriptions` directory.

Run this notebook with the ADRQ kernel.

In [10]:
# Setup: paths to the source repo and the Rainfall Rescue monthly SQLite database.
# The database itself is built in the companion notebook RR_data_ingest.ipynb.

import os
from pathlib import Path
import sqlite3

from src.rainfall_rescue_sqlite.ingest import default_db_path

repo_dir = f"{os.getenv('PDIR')}/Rainfall-Rescue"
rainfall_rescue_root = Path(repo_dir)
db_path = default_db_path()

print("RR root dir:", rainfall_rescue_root)
print("DB path:", db_path)

from src.rainfall_rescue_sqlite.ensemble_ingest import (
    default_ensemble_db_path,
    default_ensemble_root,
    ingest_ensemble_json,
)

# Parameter: root directory holding the ensemble transcription JSON files.
# Edit this to point at your source data. Falls back to the package default
# (or the ENSEMBLE_TRANSCRIPTIONS_ROOT environment variable) when left as None.
ensemble_root = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions"

if ensemble_root is None:
    ensemble_root = default_ensemble_root()
else:
    ensemble_root = Path(ensemble_root)

ensemble_db_path = default_ensemble_db_path()

print("Ensemble root dir:", ensemble_root)
print("Ensemble DB path:", ensemble_db_path)


RR root dir: /data/scratch/philip.brohan/ADRQ/Rainfall-Rescue
DB path: /data/scratch/philip.brohan/ADRQ/Rainfall-Rescue/rainfall_rescue.sqlite
Ensemble root dir: /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions
Ensemble DB path: /data/scratch/philip.brohan/ADRQ/ensemble_transcriptions.sqlite


In [11]:
# Rebuild ensemble transcription database from scratch. 
# This version runs the ingest in a single thread.
# Use this with max_files set for a quick smoke run, e.g. max_files=50.
# Will work for the full dataset, but takes 5.5 hours for the sample dataset (50,000 pages) - would be all day for everything.
# Use the cluster (cell below) for the full run.

ensemble_result = ingest_ensemble_json(
    ensemble_root=ensemble_root,
    db_path=ensemble_db_path,
    max_files=50,
)

ensemble_result

KeyboardInterrupt: 

## Full-scale ingest with SLURM

The in-process ingest above parses every JSON file in a single Python process,
which takes ~5.5 hours for the sample dataset. Because each JSON file is
self-contained, the work is embarrassingly parallel: we split the (sorted) file
list into shards and run them as a SLURM array job on the SPICE cluster.

**How the work is sharded**

- The discovered JSON files are divided into `ENSEMBLE_NUM_SHARDS` (default 100)
  interleaved slices (`files[shard_index::num_shards]`). Each shard parses its
  own slice and writes a complete ensemble database for just those files to a
  private SQLite file (`ens_shard_XXXXX.sqlite`).
- A final **merge** step reads all the shard files and combines them into
  `ensemble_transcriptions.sqlite`, offsetting each shard's `file_id` values so
  the `ensemble_daily_values` / `ensemble_monthly_totals` foreign keys stay
  consistent.

**Pipeline (two stages, chained with `--dependency=afterok`)**

| Stage | Script | SLURM | Purpose |
|-------|--------|-------|---------|
| ingest | `scripts/slurm/ingest_ensemble_array.sbatch` | array `0-99` | parse each shard's JSON slice in parallel |
| merge  | `scripts/slurm/merge_ensemble_shards.sbatch` | 1 job | combine shards into one database |

Shard count, resources and an optional file cap are set in
`scripts/slurm/config.sh` (`ENSEMBLE_NUM_SHARDS`, `ENSEMBLE_MAX_FILES`, and the
`EINGEST_*` / `EMERGE_*` resource variables).

**Cluster-filesystem note.** As with the matching pipeline, SQLite can't do its
POSIX write-locking on the shared parallel filesystem, so every database is
written on node-local `$TMPDIR` and copied back to shared disc
(`src/rainfall_rescue_sqlite/sqlite_staging.py`); the JSON source tree is only
ever read.

**Setting the source directory.** The `ensemble_root` variable set in the
notebook cell above is *not* seen by the SLURM jobs — the cluster path is chosen
independently. To point the pipeline at your JSON tree, set `ENSEMBLE_ROOT`
(defined in `scripts/slurm/config.sh`, and also accepting the legacy
`ENSEMBLE_TRANSCRIPTIONS_ROOT`) when you launch the submit script. It is passed
*explicitly* to every job via `--ensemble-root`, so it can never silently fall
back to the sample. If left unset, the package default (the operational
**sample**, ~46k files) is used — so always set it for a full run.

**Launch the whole pipeline** (from the repository root, on a login node):

ENSEMBLE_ROOT=/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions \
ENSEMBLE_TRANSCRIPTIONS_ROOT=/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions \
  scripts/slurm/submit_ensemble_ingest.sh
```

Quick test on a small slice (500 files across the shards):

ENSEMBLE_ROOT=/path/to/ensemble_transcriptions \
ENSEMBLE_TRANSCRIPTIONS_ROOT=/path/to/ensemble_transcriptions \
ENSEMBLE_MAX_FILES=500 scripts/slurm/submit_ensemble_ingest.sh
```

**Monitor it**

```bash
squeue -u $USER                                              # queued / running
sacct --format=JobID%15,JobName%14,State,ExitCode,Elapsed   # final states
ls -t $PDIR/slurm_logs | head                               # per-job logs
```

With 100 shards the ~5.5-hour serial parse drops to a few minutes per shard plus
a short merge. The cells below inspect the resulting `ensemble_transcriptions.sqlite`.

In [14]:
# Basic check of the ensemble database contents after ingestion.
# Reads metadata from ingest_runs table instead of full table scans (~1ms instead of 40 mins).

with sqlite3.connect(f"file:{ensemble_db_path}?immutable=1", uri=True) as ens_conn:
    ens_cur = ens_conn.cursor()
    
    run = ens_cur.execute(
        """
        SELECT files_discovered, files_ingested, daily_rows, total_rows, errors, status
        FROM ensemble_ingestion_runs
        ORDER BY run_id DESC
        LIMIT 1
        """
    ).fetchone()
    
    if run:
        discovered, ingested, daily, totals, errors, status = run
        print(f"Latest ingest run: {status}")
        print(f"  Files discovered: {discovered} | ingested: {ingested}")
        print(f"  Daily rows: {daily} | Total rows: {totals}")
        print(f"  Errors: {errors}")
        if status != "success" and errors > 0:
            print("  ⚠️ Ingest completed with errors; see ensemble_ingestion_file_errors")
    else:
        print("No ingest run found in database")

Latest ingest run: success
  Files discovered: 584513 | ingested: 584513
  Daily rows: 1087194180 | Total rows: 35070780
  Errors: 0


In [ ]:
# View any parse errors from the latest ingest run
with sqlite3.connect(f"file:{ensemble_db_path}?immutable=1", uri=True) as ens_conn:
    ens_conn.row_factory = sqlite3.Row
    ens_cur = ens_conn.cursor()
    
    # Get latest run_id
    run_id = ens_cur.execute(
        "SELECT run_id FROM ensemble_ingestion_runs ORDER BY run_id DESC LIMIT 1"
    ).fetchone()[0]
    
    errors = ens_cur.execute(
        """
        SELECT source_path, error_message
        FROM ensemble_ingestion_file_errors
        WHERE run_id = ?
        ORDER BY source_path
        """,
        (run_id,)
    ).fetchall()
    
    if errors:
        print(f"Found {len(errors)} parse errors in run {run_id}:\n")
        for err in errors:
            print(f"  {err['source_path']}")
            print(f"    → {err['error_message']}\n")
    else:
        print(f"No errors recorded for run {run_id}")

Found 50 parse errors in run 1:

  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/DRain_1951-1962_RainNos_2353-2373-389.json
    → Entry 0 under 'Day 1' must have 5 values, found 1

  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/DRain_1951-1962_RainNos_2353-2373-39.json
    → Entry 0 under 'Day 1' must have 5 values, found 1

  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/DRain_1951-1962_RainNos_2353-2373-390.json
    → Entry 0 under 'Day 1' must have 5 values, found 1

  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/DRain_1951-1962_RainNos_2353-2373-391.json
    → Entry 0 under 'Day 1' must have 5 values, found 1

  /data/scratch/philip.brohan/documents/Daily_Rainfall_UK/operational_full/ensemble_transcriptions/DRain_1951-1962_RainNos_2353-2373-392.json
    → Entry 0 under 'Day 1' mu

In [15]:
# Sample daily ensemble values for one file (day 1, all months and members)

with sqlite3.connect(ensemble_db_path) as ens_conn:
    ens_conn.row_factory = sqlite3.Row
    ens_cur = ens_conn.cursor()

    file_row = ens_cur.execute(
        "SELECT file_id, file_name, year_start, year_end, descriptor, section_id, num_days "
        "FROM ensemble_files ORDER BY file_name LIMIT 1"
    ).fetchone()

    print(
        f"file_id={file_row['file_id']}  {file_row['file_name']}  "
        f"years={file_row['year_start']}-{file_row['year_end']}  "
        f"descriptor={file_row['descriptor']}  section={file_row['section_id']}  "
        f"days={file_row['num_days']}"
    )

    daily_sample = ens_cur.execute(
        """
        SELECT month, ensemble_member, rainfall
        FROM ensemble_daily_values
        WHERE file_id = ? AND day_of_month = 1
        ORDER BY month, ensemble_member
        """,
        (file_row["file_id"],),
    ).fetchall()

for r in daily_sample[:15]:
    print(f"  month={r['month']:>2}  member={r['ensemble_member']}  rainfall={r['rainfall']}")

file_id=1  DRain_1861-1870_Alderney-0.json  years=None-None  descriptor=None  section=None  days=31
  month= 1  member=1  rainfall=None
  month= 1  member=2  rainfall=None
  month= 1  member=3  rainfall=None
  month= 1  member=4  rainfall=None
  month= 1  member=5  rainfall=None
  month= 2  member=1  rainfall=None
  month= 2  member=2  rainfall=None
  month= 2  member=3  rainfall=None
  month= 2  member=4  rainfall=None
  month= 2  member=5  rainfall=None
  month= 3  member=1  rainfall=None
  month= 3  member=2  rainfall=None
  month= 3  member=3  rainfall=None
  month= 3  member=4  rainfall=None
  month= 3  member=5  rainfall=None


In [ ]:
# Monthly totals for the same file: mean ensemble total per month

with sqlite3.connect(ensemble_db_path) as ens_conn:
    ens_conn.row_factory = sqlite3.Row
    ens_cur = ens_conn.cursor()

    totals = ens_cur.execute(
        """
        SELECT month,
               AVG(total) AS mean_total,
               COUNT(total) AS n_values
        FROM ensemble_monthly_totals
        WHERE file_id = ?
        GROUP BY month
        ORDER BY month
        """,
        (file_row["file_id"],),
    ).fetchall()

for r in totals:
    mean_total = r["mean_total"]
    mean_str = f"{mean_total:.3f}" if mean_total is not None else "n/a"
    print(f"  month={r['month']:>2}  mean_total={mean_str:>8}  members_with_value={r['n_values']}")

  month= 1  mean_total=   4.410  members_with_value=4
  month= 2  mean_total=   3.459  members_with_value=5
  month= 3  mean_total=   1.798  members_with_value=5
  month= 4  mean_total=   1.602  members_with_value=5
  month= 5  mean_total=   2.565  members_with_value=5
  month= 6  mean_total=   1.523  members_with_value=5
  month= 7  mean_total=   0.992  members_with_value=5
  month= 8  mean_total=   1.104  members_with_value=5
  month= 9  mean_total=   0.394  members_with_value=5
  month=10  mean_total=   3.581  members_with_value=5
  month=11  mean_total=   3.576  members_with_value=5
  month=12  mean_total=   3.255  members_with_value=5
